# Unidad 6. 🧪 Análisis automático de sesgo léxico y discursivo

Cuando entregues tu cuaderno, debes nombrar el fichero (.ipynb) de la siguiente manera:

(3 últimos dígitos del alumno 1)_(3 últimos dígitos del alumno 2)_(UD+número)

Por ejemplo:
- Dos alumnos entregan las actividades evaluables de la Unidad 6
- Identificador alumno 1: 200800331
- Identificador alumno 2: 200800229

Nombre del fichero: `331_229_UD6.ipynb`

Este cuaderno continúa el trabajo con textos reales y tareas profesionales de PLN, avanzando hacia el análisis crítico del lenguaje mediante programación.

## 📌 1. Objetivos de aprendizaje

Al finalizar este cuaderno, el alumnado será capaz de:

- Comprender qué se entiende por sesgo lingüístico desde una perspectiva computacional

- Implementar un análisis automático básico de sesgos léxicos

- Interpretar críticamente los resultados obtenidos

- Reflexionar sobre los límites éticos y lingüísticos del PLN

## 🧠 2. Contexto profesional

**Situación profesional**

Trabajas como lingüista computacional en una empresa que desarrolla herramientas de análisis textual para medios de comunicación y redes sociales.

Un cliente solicita una auditoría lingüística de varios textos para detectar:

- lenguaje excluyente

- sesgos de género

- léxico valorativo excesivo

Tu tarea consiste en diseñar un prototipo de análisis automático, sabiendo que los resultados deberán ser interpretados por especialistas humanos.

## 📚 3. Marco conceptual

**3.1 ¿Qué es el sesgo lingüístico?**

Desde un punto de vista lingüístico, el sesgo puede manifestarse como:

- Uso sistemático de ciertas palabras frente a otras

- Invisibilización de colectivos

- Carga valorativa (positiva o negativa)

- Marcos discursivos recurrentes

Desde el PLN:

- No se “detecta intención”

- Se identifican patrones léxicos y frecuencias

**3.2 Tipos de sesgo abordables computacionalmente**

En este cuaderno trabajaremos con:

- Sesgo de género (léxico marcado)

- Lenguaje valorativo

- Polaridad léxica básica

⚠️ Importante: No trabajamos con modelos de ML ni IA generativa, sino con reglas explícitas y transparentes.

## 🛠️ 4. Preparación del entorno

In [ ]:
# Si no lo tenemos instalado todavía: pip install pandas

In [1]:
import pandas as pd
import re
from collections import Counter

## 📄 5. Corpus de trabajo

En vez de depender de descargas externas (que a veces fallan en clase), trabajaremos con un **mini‑corpus en español incluido en el cuaderno**. Está pensado para que aparezcan **marcas de género**, **léxico valorativo** y expresiones típicas de titulares / redes.

> Si tu entorno tiene internet, al final te dejo también una **opción opcional** para cargar un dataset real con `datasets` (Hugging Face) y comparar.


El objetivo didáctico es doble:

1) Que los resultados salgan **siempre** y sean **interpretables**.
2) Que el alumnado pueda **cambiar/añadir** textos y léxicos y ver cómo cambian los patrones.

Trabajaremos con un corpus corto pero “jugoso”, con frases que simulan titulares, comentarios y mensajes breves.


In [2]:
# Dependencias (ligeras) para que funcione en cualquier aula
import pandas as pd
import re
from collections import Counter, defaultdict

# Opcional: si tienes spaCy instalado y el modelo en español disponible,
# lo usaremos para lematizar mejor. Si no, el cuaderno sigue funcionando.
try:
    import spacy
    try:
        nlp = spacy.load("es_core_news_sm")
    except Exception:
        nlp = None
except Exception:
    spacy = None
    nlp = None

print("spaCy disponible:", spacy is not None, "| Modelo es_core_news_sm cargado:", nlp is not None)

from IPython.display import display


spaCy disponible: True | Modelo es_core_news_sm cargado: True


In [3]:
# Mini‑corpus didáctico (puedes ampliarlo en clase)
corpus = [
    # Titulares / estilo prensa
    "La candidata promete una reforma ambiciosa y recibe apoyo masivo.",
    "El candidato es criticado por su discurso agresivo y sus promesas vagas.",
    "Una mujer lidera el proyecto con resultados excelentes, según el informe.",
    "Un hombre asume el cargo y la oposición lo califica de incompetente.",
    "Las científicas logran un avance brillante; algunos medios lo minimizan.",
    "Los científicos presentan datos sólidos y la prensa lo celebra como histórico.",
    "La madre trabajadora es descrita como sacrificada y ejemplar.",
    "El padre trabajador es retratado como líder nato y decidido.",
    "La jefa fue llamada 'mandona' por exigir puntualidad y rigor.",
    "El jefe fue descrito como 'exigente' por exigir puntualidad y rigor.",
    # Redes / comentarios
    "Esa chica es genial, súper lista, pero demasiado emocional para el puesto.",
    "Ese chico es genial, súper listo, y con carácter fuerte para liderar.",
    "La ministra actuó con calma y firmeza; la llamaron fría y calculadora.",
    "El ministro actuó con calma y firmeza; lo llamaron responsable y sereno.",
    "Las inmigrantes fueron descritas como una amenaza; comentario lamentable.",
    "Los inmigrantes fueron descritos como una amenaza; comentario lamentable.",
    "La deportista tuvo un rendimiento espectacular; aun así la juzgan por su ropa.",
    "El deportista tuvo un rendimiento espectacular; aun así lo juzgan por su actitud.",
    "Mi compañera es competente y trabajadora, pero dicen que manda demasiado.",
    "Mi compañero es competente y trabajador, y dicen que tiene buen liderazgo.",
    # Educación / aula
    "Las alumnas participaron activamente y el profesor dijo que eran aplicadas.",
    "Los alumnos participaron activamente y el profesor dijo que eran brillantes.",
    "La estudiante recibió un comentario negativo: 'hablas mucho'.",
    "El estudiante recibió un comentario negativo: 'interrumpes demasiado'.",
    # Valorativo fuerte
    "Un informe terrible acusa al directivo de gestión desastrosa.",
    "Una gestión excelente reduce el estrés y mejora el ambiente de trabajo.",
    "La campaña fue un éxito rotundo, con un mensaje positivo y claro.",
    "La campaña fue un desastre absoluto, con un mensaje confuso y agresivo.",
    # Neutras (para contraste)
    "El equipo presentó el proyecto y respondió preguntas del público.",
    "La reunión terminó a tiempo y se enviaron las actas por correo."
]

df = pd.DataFrame({'text': corpus})
df.head()


,text
0,La candidata promete una reforma ambiciosa y r...
1,El candidato es criticado por su discurso agre...
2,Una mujer lidera el proyecto con resultados ex...
3,Un hombre asume el cargo y la oposición lo cal...
4,Las científicas logran un avance brillante; al...


In [4]:
# Tamaño del corpus y vista rápida
print("Nº de textos:", len(df))
df.sample(5, random_state=1)


Nº de textos: 30


,text
17,El deportista tuvo un rendimiento espectacular...
21,Los alumnos participaron activamente y el prof...
10,"Esa chica es genial, súper lista, pero demasia..."
19,"Mi compañero es competente y trabajador, y dic..."
14,Las inmigrantes fueron descritas como una amen...


`DatasetDict`
- Es un diccionario especial de HuggingFace que agrupa varios datasets bajo diferentes nombres (keys).
- En este caso, las keys son: `train`, `test` y `validation`.

Representan los conjuntos de datos para entrenar, evaluar y validar un modelo de machine learning:

| Conjunto     | Filas (`num_rows`) | Uso principal                                          |
| ------------ | ------------------ | ------------------------------------------------------ |
| `train`      | 45,615             | Para **entrenar** tu modelo.                           |
| `test`       | 12,284             | Para **evaluar** el rendimiento final del modelo.      |
| `validation` | 2,000              | Para **ajustar hiperparámetros** y evitar sobreajuste. |

`features: ['text', 'label']`
Cada dataset tiene columnas (features):
- 'text' → el contenido del tweet.
- 'label' → la etiqueta de sentimiento asociada: normalmente 0 = negativo, 1 = neutral, 2 = positivo (puede variar según la documentación de tweet_eval).

**En resumen**:
Tenemos un dataset de análisis de sentimientos de tweets dividido en entrenamiento, prueba y validación.
Cada ejemplo tiene un texto (text) y su etiqueta de sentimiento (label).
`DatasetDict` es solo la forma de HuggingFace de organizar estos tres conjuntos bajo un mismo objeto:

- `df = dataset["train"].to_pandas()`
- `df = df.sample(200, random_state=42).reset_index(drop=True)`
- `df.head()`

In [5]:
# En clase suele ser útil trabajar con un subconjunto para leer ejemplos.
# Aquí el corpus ya es pequeño, pero dejamos el patrón.
df = df.sample(min(30, len(df)), random_state=42).reset_index(drop=True)
df.head(10)


,text
0,"La campaña fue un desastre absoluto, con un me..."
1,Los inmigrantes fueron descritos como una amen...
2,El estudiante recibió un comentario negativo: ...
3,El deportista tuvo un rendimiento espectacular...
4,La jefa fue llamada 'mandona' por exigir puntu...
5,El jefe fue descrito como 'exigente' por exigi...
6,El equipo presentó el proyecto y respondió pre...
7,Un informe terrible acusa al directivo de gest...
8,La ministra actuó con calma y firmeza; la llam...
9,La candidata promete una reforma ambiciosa y r...


✅ BHP: El siguiente código extrae todos los textos de la columna text del DataFrame y los almacena como una lista. Esto es habitual en tareas de PLN, por ejemplo antes de tokenizar, vectorizar o pasar los textos a un modelo.

In [ ]:
textos = df["text"].tolist()

# df["text"] = Selecciona la columna text del DataFrame df
# .tolist() convierte esa columna en una lista de textos

## 🧩 6. Preprocesamiento del texto

### 6.1 Normalización

Incluye:

- Minúsculas

- Eliminación de URLs

- Eliminación de menciones (@)

- Eliminación de hashtags (manteniendo el contenido)

- Eliminación de números

- Normalización de espacios

In [7]:
def normalizar_texto(texto):
    texto = texto.lower()
    texto = re.sub(r"http\S+|www\S+", "", texto)  # URLs
    texto = re.sub(r"@\w+", "", texto)            # menciones
    texto = re.sub(r"#", "", texto)               # hashtags
    texto = re.sub(r"\d+", "", texto)             # números
    texto = re.sub(r"[^\w\s]", "", texto)         # puntuación
    texto = re.sub(r"\s+", " ", texto).strip()    # espacios
    return texto

# Aplicamos
textos_norm = [normalizar_texto(t) for t in textos]


Atención: Esta normalización no es “neutra”: cada decisión implica una hipótesis lingüística sobre qué información es relevante y cuál no.

### 6.2 Tokenización y lematización con SpaCy

In [8]:
# Tokenización y (opcionalmente) lematización
# - Si spaCy + un modelo de español están disponibles: mejor lematización y stopwords.
# - Si no: tokenización por regex + stopwords estándar (spaCy o NLTK si están disponibles).
#
# Nota didáctica: usar stopwords "oficiales" evita listas manuales y hace el análisis más reproducible.
#
# ✅ Importante: distintos datasets usan nombres de columna distintos (texto, text, tweet, etc.).
#    Para que el cuaderno sea robusto, detectamos automáticamente la columna de texto.

import re

def get_stopwords_es():
    """Devuelve un conjunto de stopwords en español.
    Prioridad: spaCy (lista integrada) → NLTK (lista estándar) → fallback mínimo.
    """
    # 1) spaCy
    try:
        from spacy.lang.es.stop_words import STOP_WORDS as SPACY_STOP
        return set(SPACY_STOP)
    except Exception:
        pass

    # 2) NLTK
    try:
        import nltk
        from nltk.corpus import stopwords
        try:
            return set(stopwords.words("spanish"))
        except LookupError:
            # Intenta descargar solo si falta (puede fallar sin internet; lo gestionamos).
            try:
                nltk.download("stopwords", quiet=True)
                return set(stopwords.words("spanish"))
            except Exception:
                pass
    except Exception:
        pass

    # 3) Fallback mínimo (solo si no hay librerías disponibles)
    return {
        "de","la","que","el","en","y","a","los","del","se","las","por","un","para","con","no","una","su","al","lo",
        "como","más","pero","sus","le","ya","o","este","sí","porque","esta","entre","cuando","muy","sin","sobre","también"
    }

STOPWORDS_ES = get_stopwords_es()

def detectar_columna_texto(df):
    """Heurística simple para encontrar la columna que contiene el texto."""
    candidatos = ["texto", "text", "tweet", "tweets", "content", "sentence", "review"]
    for c in candidatos:
        if c in df.columns:
            return c
    # Si no hay un nombre típico, elegimos la primera columna de tipo 'object' (texto)
    obj_cols = [c for c in df.columns if df[c].dtype == "object"]
    if obj_cols:
        return obj_cols[0]
    raise KeyError("No he encontrado una columna de texto en el DataFrame. Revisa df.columns")

TEXT_COL = detectar_columna_texto(df)

def tokenizar(texto: str):
    # Mantiene letras con acentos/ñ y elimina emojis/símbolos.
    return re.findall(r"[a-záéíóúüñ]+", str(texto).lower())

def tokenizar_y_lematizar(texto: str):
    # Si no hay spaCy, aplicamos tokenización + stopwords.
    if nlp is None:
        toks = tokenizar(texto)
        return [t for t in toks if t not in STOPWORDS_ES and len(t) > 2]

    # Con spaCy, usamos lemas y stopwords del propio modelo/lista.
    doc = nlp(str(texto).lower())
    return [
        t.lemma_
        for t in doc
        if (not t.is_stop) and (not t.is_space) and (t.is_alpha) and len(t.lemma_) > 2
    ]

tokens = [tokenizar_y_lematizar(t) for t in df[TEXT_COL].tolist()]
df["tokens"] = tokens

# Vista rápida
df[[TEXT_COL, "tokens"]].head()


,text,tokens
0,"La campaña fue un desastre absoluto, con un me...","[campaña, desastre, absoluto, mensaje, confuso..."
1,Los inmigrantes fueron descritos como una amen...,"[inmigrante, describir, amenaza, comentario, l..."
2,El estudiante recibió un comentario negativo: ...,"[estudiante, recibir, comentario, negativo, in..."
3,El deportista tuvo un rendimiento espectacular...,"[deportista, rendimiento, espectacular, juzgar..."
4,La jefa fue llamada 'mandona' por exigir puntu...,"[jefa, llamar, mandona, exigir, puntualidad, r..."


In [ ]:
# (Ya cargamos spaCy arriba si está disponible.)


In [ ]:
# (Ya tokenizamos y lematizamos arriba.)


## 📑 7. Listas léxicas de sesgo

Vamos a crearnos nuestras propias listas léxicas a partir de repositorios en español:

### 7.1 Léxico de género marcado

In [9]:
# Diccionario de pares de género (muy ampliable en clase)
dic_genero = {
    "hombre": "mujer",
    "mujer": "hombre",
    "chico": "chica",
    "chica": "chico",
    "niño": "niña",
    "niña": "niño",
    "padre": "madre",
    "madre": "padre",
    "jefe": "jefa",
    "jefa": "jefe",
    "ministro": "ministra",
    "ministra": "ministro",
    "candidato": "candidata",
    "candidata": "candidato",
    "científico": "científica",
    "científica": "científico",
    "alumno": "alumna",
    "alumna": "alumno",
    "estudiante": "estudiante",  # neutral, lo dejamos para demostrar limitaciones
    "deportista": "deportista"
}

lexico_genero = sorted(set(dic_genero.keys()) | set(dic_genero.values()))
lexico_genero[:20], len(lexico_genero)


(['alumna',
  'alumno',
  'candidata',
  'candidato',
  'chica',
  'chico',
  'científica',
  'científico',
  'deportista',
  'estudiante',
  'hombre',
  'jefa',
  'jefe',
  'madre',
  'ministra',
  'ministro',
  'mujer',
  'niña',
  'niño',
  'padre'],
 20)

### 7.2 Léxico valorativo

In [10]:
# Léxico valorativo (ampliado) con polaridad sencilla
lexico_valorativo_positivo = [
    "excelente","buen","bueno","maravilloso","fantástico","positivo","agradable","genial","estupendo",
    "feliz","amor","brillante","sólido","historico","histórico","éxito","rotundo","claro","sereno","responsable",
    "espectacular","ambicioso","masivo","mejora","mejor","apoyo"
]

lexico_valorativo_negativo = [
    "malo","horrible","terrible","negativo","desagradable","agresivo","agresiva","odio","peor","lamentable",
    "incompetente","vago","vagas","minimiza","fría","frio","calculadora","mandona","desastre","desastrosa",
    "confuso","confusa","acus","acusa","amenaza"
]

dic_valorativo = {w.lower(): "positivo" for w in lexico_valorativo_positivo}
dic_valorativo.update({w.lower(): "negativo" for w in lexico_valorativo_negativo})

print("Tamaño diccionario valorativo:", len(dic_valorativo))


Tamaño diccionario valorativo: 51


## 🔍 8. Detección automática de sesgo léxico

### 8.1 Recuento en corpus

In [11]:
def detectar_lexico_genero(tokens_doc, lexico_genero):
    return [t for t in tokens_doc if t in lexico_genero]

def detectar_lexico_valorativo(tokens_doc, dic_valorativo):
    return [t for t in tokens_doc if t in dic_valorativo]

resultados = []
for i, texto_tokens in enumerate(tokens):
    genero = detectar_lexico_genero(texto_tokens, lexico_genero)
    valorativo = detectar_lexico_valorativo(texto_tokens, dic_valorativo)

    score_valorativo = sum(1 if dic_valorativo[t] == "positivo" else -1 for t in valorativo)

    resultados.append({
        "texto_id": i + 1,
        "texto": textos[i],
        "tokens": texto_tokens,
        "lexico_genero": genero,
        "n_genero": len(genero),
        "lexico_valorativo": valorativo,
        "n_valorativo": len(valorativo),
        "score_valorativo": score_valorativo
    })

df_resultados = pd.DataFrame(resultados)
df_resultados.head()


,texto_id,texto,tokens,lexico_genero,n_genero,lexico_valorativo,n_valorativo,score_valorativo
0,1,"La campaña fue un desastre absoluto, con un me...","[campaña, desastre, absoluto, mensaje, confuso...",[],0,"[desastre, confuso, agresivo]",3,-3
1,2,Los inmigrantes fueron descritos como una amen...,"[inmigrante, describir, amenaza, comentario, l...",[],0,"[amenaza, lamentable]",2,-2
2,3,El estudiante recibió un comentario negativo: ...,"[estudiante, recibir, comentario, negativo, in...",[estudiante],1,[negativo],1,-1
3,4,El deportista tuvo un rendimiento espectacular...,"[deportista, rendimiento, espectacular, juzgar...",[deportista],1,[espectacular],1,1
4,5,La jefa fue llamada 'mandona' por exigir puntu...,"[jefa, llamar, mandona, exigir, puntualidad, r...",[jefa],1,[mandona],1,-1


**✅ BHP**

📌 Qué representa este resultado: `df_resultados` resume, para cada texto del minicorpus didáctico, el resultado de aplicar:
- un léxico de género (términos como jefa, estudiante, deportista…)
- un léxico valorativo explícito, con polaridad codificada: "positivo" → +1 // "negativo" → −1
Aquí sí estamos forzando claridad, a diferencia del corpus real.

**🧱 Interpretación fila por fila (lo esencial)**

🟢 Fila 1 > Texto: “La campaña fue un desastre absoluto, con un mensaje confuso y agresivo.”
- lexico_genero: [] → no hay referencias a personas/género
- lexico_valorativo: [desastre, confuso, agresivo]
- score_valorativo: −3

👉 Ejemplo “limpio” de valoración negativa explícita. --> Sirve para mostrar cómo el método captura bien adjetivos evaluativos, incluso sin sujetos humanos.

🟢 Fila 2 > Texto: “Los inmigrantes fueron descritos como una amenaza; comentario lamentable.”
- lexico_genero: [] (según tu léxico, inmigrantes no cuenta como género)
- lexico_valorativo: [amenaza, lamentable]
- score: −2

👉 Buen ejemplo de discurso problemático capturado léxicamente, valoración negativa clara, sin ambigüedad. Muy potente para discusión ética y discursiva.

🟡 Fila 3 > Texto: “El estudiante recibió un comentario negativo: ‘interrumpes demasiado’.”
- lexico_genero: [estudiante]
- lexico_valorativo: [negativo]
- score: −1

👉 Aquí vemos coexistencia de referencia a persona + evaluación explícita, valoración moderada (no extrema). Sirve para mostrar cómo el score no es binario, sino gradual.

🟢 Fila 4 > Texto: “El deportista tuvo un rendimiento espectacular…”
- lexico_genero: [deportista]
- lexico_valorativo: [espectacular]
- score: +1

👉 Ejemplo claro de valoración positiva explícita, asociada a un sujeto. Muy útil para contrastar con el anterior.

🔴 Fila 5 > Texto: “La jefa fue llamada ‘mandona’…”
- lexico_genero: [jefa]
- lexico_valorativo: [mandona]
- score: −1

👉 Este es uno de los ejemplos didácticamente más importantes del cuaderno:
- mismo comportamiento (“exigir puntualidad”)
- evaluación negativa
- término con carga de género

Aquí el método detecta la palabra, pero no entiende el sesgo → eso lo hace el análisis humano.

Con esta tabla puedes decir en clase:
- “Aquí vemos cómo un análisis léxico capta bien la valoración cuando esta es explícita.”
- “Cuando pasamos a datos reales, el mismo método revela sus límites.”
- “Por eso el PLN no sustituye a la interpretación, la apoya.”

## 📊 9. Análisis agregado - frecuencias globales

### 9.1. Frecuencias globales de los términos de interés

In [12]:
todos_tokens = [t for sublist in df_resultados["tokens"].tolist() for t in sublist]
frecuencias = Counter(todos_tokens)

frecuencias_sesgo = pd.DataFrame([
    {"palabra": p, "frecuencia": frecuencias[p],
     "tipo": ("género" if p in lexico_genero else "valorativo"),
     "polaridad": (dic_valorativo.get(p, ""))}
    for p in sorted(set(lexico_genero) | set(dic_valorativo.keys()))
    if frecuencias[p] > 0
]).sort_values(["tipo","frecuencia"], ascending=[True, False])

frecuencias_sesgo


,palabra,frecuencia,tipo,polaridad
15,deportista,2,género,
18,estudiante,2,género,
1,alumna,1,género,
2,alumno,1,género,
8,candidata,1,género,
9,candidato,1,género,
10,chica,1,género,
11,chico,1,género,
12,científica,1,género,
13,científico,1,género,


**✅ BHP:**

📊 Qué es esta tabla (recordatorio rápido): `frecuencias_sesgo recoge`, en todo el minicorpus, la frecuencia de aparición de:
- términos del léxico de género
- términos del léxico valorativo, con su polaridad asociada
Es decir: ya no miramos textos individuales, sino el patrón global del corpus.

**🧱 Interpretación por bloques**

1️⃣ Términos de género > Ejemplos más frecuentes:
- deportista (2)
- estudiante (2)
- El resto aparece una sola vez: candidata / candidato, jefa / jefe, madre / padre, ministra / ministro, alumna / alumno, científica / científico, mujer / hombre

Interpretación: El minicorpus está deliberadamente equilibrado (masculino / femenino, singular / plural) y con roles comparables. No hay una dominancia clara de un término sobre otro.

“Aquí no vemos sesgo por frecuencia, porque el corpus está diseñado para contrastar usos.”

2️⃣ Términos valorativos más frecuentes > Con frecuencia 2 aparecen:
- Negativos: agresivo, amenaza, lamentable, negativo
- Positivos: brillante, espectacular, excelente, genial

Interpretación: El corpus contiene valoración explícita y balanceada. Hay tanto positivos como negativos. No domina un solo polo

Esto confirma que: el léxico está bien elegido, el método captura bien valoración explícita.

**✔️ Qué muestra este análisis**

- El método detecta qué palabras valorativas circulan
- Permite ver qué tipos de evaluación predominan
- Hace visibles asimetrías potenciales: mandona vs responsable, fría vs sereno, calculadora vs estratégico (implícito)

Aunque el método:
- ❌ no entiende ironía
- ❌ no entiende contexto

PREGUNTAS:

- ¿Por qué mandona aparece solo una vez, pero es tan relevante?
- ¿Qué diferencia hay entre fría y sereno?
- ¿Son todos los términos “negativos” igualmente negativos?
- ¿Qué no estamos capturando con este enfoque?

### 📌 Ejemplos “interesantes” para discutir

In [13]:
# 1) Textos con más marcas de género
top_genero = df_resultados.sort_values(["n_genero","n_valorativo"], ascending=False).head(8)[
    ["texto_id","n_genero","lexico_genero","n_valorativo","lexico_valorativo","score_valorativo","texto"]
]
top_genero


,texto_id,n_genero,lexico_genero,n_valorativo,lexico_valorativo,score_valorativo,texto
9,10,1,[candidata],3,"[ambicioso, apoyo, masivo]",3,La candidata promete una reforma ambiciosa y r...
8,9,1,[ministra],2,"[fría, calculadora]",-2,La ministra actuó con calma y firmeza; la llam...
12,13,1,[científico],2,"[sólido, histórico]",2,Los científicos presentan datos sólidos y la p...
13,14,1,[ministro],2,"[responsable, sereno]",2,El ministro actuó con calma y firmeza; lo llam...
16,17,1,[candidato],2,"[agresivo, vago]",-2,El candidato es criticado por su discurso agre...
2,3,1,[estudiante],1,[negativo],-1,El estudiante recibió un comentario negativo: ...
3,4,1,[deportista],1,[espectacular],1,El deportista tuvo un rendimiento espectacular...
4,5,1,[jefa],1,[mandona],-1,La jefa fue llamada 'mandona' por exigir puntu...


**✅ BHP:**

📌 Qué está mostrando esta tabla: `top_genero selecciona` los 8 textos del minicorpus que contienen referencias explícitas de género (n_genero), y, en caso de empate, más léxico valorativo (n_valorativo).

👉 No son “los más frecuentes”, sino los más densos discursivamente según tus criterios.

**🧱 Interpretación fila por fila (agrupando patrones)**

🟢 1. Evaluación positiva asociada a figuras masculinas y femeninas > Ejemplos:
- Candidata → ambiciosa, apoyo, masivo → +3
- Científico → sólido, histórico → +2
- Ministro → responsable, sereno → +2

Interpretación: El método captura valoración positiva explícita. Asociada a: competencia, liderazgo, éxito institucional

👉 Aquí el enfoque léxico funciona “como en el manual”.

🔴 2. Evaluación negativa con carga de género > Ejemplos claros:
- Ministra → fría, calculadora → −2
- Candidato → agresivo, vago → −2
- Jefa → mandona → −1

Interpretación: La valoración negativa aparece como crítica de carácter, no solo de resultados. En jefa, el término es claramente marcado por género

👉 Este es un punto clave para análisis crítico.

🟡 3. Casos “neutrales” o moderados
- Estudiante → negativo → −1
- Deportista → espectacular → +1

Sirven para: mostrar escalas y evitar lecturas binarias.

**✔️ Qué permite ver este ranking**

Este ranking hace visibles asimetrías léxicas (mandona vs responsable, fría vs sereno), diferencias en el tipo de evaluación (competencia técnica (sólido), carácter (fría), actitud (agresivo)). 

Aunque el método no entiende contexto, ni intencionalidad, sí permite localizar sistemáticamente textos con potencial sesgo discursivo.

In [14]:
# 2) Concordancias (KWIC) simples: contexto alrededor de un término
def kwic(tokens_corpus, term, window=4, max_lines=12):
    lines=[]
    for doc_id, toks in enumerate(tokens_corpus, start=1):
        for i,t in enumerate(toks):
            if t==term:
                left=" ".join(toks[max(0,i-window):i])
                right=" ".join(toks[i+1:i+1+window])
                lines.append({"texto_id":doc_id, "izq":left, "term":term, "dcha":right})
    return pd.DataFrame(lines).head(max_lines)

# Prueba con un término (cámbialo en clase)
kwic(tokens, term="mujer", window=5)


,texto_id,izq,term,dcha
0,18,,mujer,liderar proyecto resultado excelente informe


**✅ BHP:**

📌 Qué representa este resultado > La tabla es una concordancia KWIC (Key Word In Context) construida sobre los tokens ya preprocesados del minicorpus.

Cada fila muestra:
- texto_id: identificador del texto donde aparece el término
- izq: palabras a la izquierda del término
- term: el término buscado (mujer)
- dcha: palabras a la derecha del término

En tu caso aparece una sola ocurrencia.

Qué nos dice esto:
1. El término mujer aparece una sola vez en todo el minicorpus tokenizado. Esto es coherente con el diseño del corpus (controlado y pequeño).
2. No hay contexto a la izquierda (izq vacío): porque mujer aparece al inicio del texto original: “Una mujer lidera el proyecto con resultados excelentes…”
3. El contexto a la derecha es claramente positivo: liderar, proyecto, resultado, excelente

informe 👉 Aunque el KWIC no “entiende” el significado, permite ver de un vistazo que: la referencia a “mujer” aparece en un contexto de liderazgo y evaluación positiva.



In [15]:
# 3) Co-ocurrencias: ¿qué palabras aparecen cerca de 'mujer' vs 'hombre'?
def coocurrencias(tokens_corpus, target, window=4):
    co = Counter()
    for toks in tokens_corpus:
        for i, t in enumerate(toks):
            if t == target:
                left = toks[max(0, i-window):i]
                right = toks[i+1:i+1+window]
                for w in left + right:
                    if w != target:
                        co[w] += 1
    return co

for term in ["mujer", "hombre", "jefa", "jefe"]:
    print("\nTop co-ocurrencias para:", term)
    co = coocurrencias(tokens, term, window=5).most_common(10)
    display(pd.DataFrame(co, columns=["palabra","freq"]))



Top co-ocurrencias para: mujer


,palabra,freq
0,liderar,1
1,proyecto,1
2,resultado,1
3,excelente,1
4,informe,1



Top co-ocurrencias para: hombre


,palabra,freq
0,asumir,1
1,cargo,1
2,oposición,1
3,calificar,1
4,incompetente,1



Top co-ocurrencias para: jefa


,palabra,freq
0,llamar,1
1,mandona,1
2,exigir,1
3,puntualidad,1
4,rigor,1



Top co-ocurrencias para: jefe


,palabra,freq
0,describir,1
1,exigente,1
2,exigir,1
3,puntualidad,1
4,rigor,1


**✅ BHP:**

📌 Qué está midiendo este análisis

El código calcula co-ocurrencias léxicas:
- para cada término objetivo (mujer, hombre, jefa, jefe),
- recoge las palabras que aparecen en una ventana de ±5 tokens,
- sobre los tokens ya normalizados del minicorpus.

Es decir: “¿Con qué vocabulario se asocian estos términos en el discurso?”

**🔍 Interpretación comparativa (lo importante)**

🟢 1. mujer vs hombre: tipos de acción y evaluación 

- mujer: Co-ocurrencias: liderar, proyecto, resultado, excelente, informe // Interpretación: léxico de acción y logro, énfasis en resultados positivos, evaluación basada en desempeño --> 👉 mujer aparece asociada a competencia demostrada.

- hombre: Co-ocurrencias: asumir, cargo, oposición, calificar, incompetente // Interpretación: léxico institucional (cargo, oposición), presencia explícita de evaluación negativa, foco en rol y conflicto, no en resultados --> 👉 hombre aparece asociado a posición y juicio externo.

🟡 2. jefa vs jefe: mismo comportamiento, distinta evaluación

- jefa: Co-ocurrencias:, llamar, mandona, exigir, puntualidad, rigor

- jefe: Co-ocurrencias:, describir, exigente, exigir, puntualidad, rigor

Interpretación clave: el comportamiento es idéntico: exigir, puntualidad, rigor // la evaluación léxica difiere: mandona → negativa y con carga de género; exigente → neutral/positiva

👉 Este es el ejemplo canónico de sesgo discursivo.

**Con estos resultados puedes decir:**

- “Las palabras no aparecen solas: aparecen en redes de asociación.”
- “El mismo comportamiento puede ser evaluado de forma distinta según el género.”
- “El PLN no prueba el sesgo, pero ayuda a localizarlo.”

## 🧠 10. Interpretación lingüística

Qué hace bien el sistema:

- Identifica patrones repetidos

- Permite análisis exploratorio

- Es transparente y explicable

Qué no hace:

- No entiende ironía

- No detecta intención

- No contextualiza pragmáticamente

## 🌐 11. El mismo análisis sobre un subcorpus real (descargable)

Hasta ahora trabajábamos con un mini‑corpus didáctico. En este bloque repetimos **el mismo pipeline** con un corpus real descargable (tweets en español “clean”), para que el alumnado vea el trabajo con datos con más “realidad”:

1) Descargar el dataset (una sola vez).  
2) Cargar un subcorpus manejable (muestreo).  
3) Crear un **subcorpus temático** (filtrado por términos de género) para que el fenómeno aparezca con suficiente frecuencia.  
4) Aplicar el pipeline y analizar: resúmenes, ejemplos, KWIC y co‑ocurrencias.

> Importante: no inventamos ejemplos; **filtramos** el dataset original.


### 📥 11.1 Descargar (una vez) un subcorpus real: TwitterSentimentDataset (tweets en español)

In [16]:
# Fuente: https://github.com/garnachod/TwitterSentimentDataset

import os
import pandas as pd
from urllib.request import urlretrieve

DATA_DIR = "data_subcorpus_real"
os.makedirs(DATA_DIR, exist_ok=True)

base = "https://raw.githubusercontent.com/garnachod/TwitterSentimentDataset/master/"
archivos = {
    "pos": "tweets_pos_clean.txt",
    "neg": "tweets_neg_clean.txt",
    "neu": "tweets_clean.txt",  # neutro/mixto (fichero general en el repo)
}

rutas = {}
for etiqueta, fname in archivos.items():
    ruta = os.path.join(DATA_DIR, fname)
    rutas[etiqueta] = ruta
    if not os.path.exists(ruta) or os.path.getsize(ruta) == 0:
        print(f"Descargando {fname} ...")
        urlretrieve(base + fname, ruta)

print("Archivos listos en:", DATA_DIR)
for k,v in rutas.items():
    print(k, "->", v)



Descargando tweets_pos_clean.txt ...
Descargando tweets_neg_clean.txt ...
Descargando tweets_clean.txt ...
Archivos listos en: data_subcorpus_real
pos -> data_subcorpus_real/tweets_pos_clean.txt
neg -> data_subcorpus_real/tweets_neg_clean.txt
neu -> data_subcorpus_real/tweets_clean.txt


**✅ BHP**

Este paso 11.1 significa 3 cosas:

1️⃣ Que el subcorpus real ya está disponible en tu ordenador:

La carpeta `data_subcorpus_real/` contiene ahora tres ficheros reales descargados desde GitHub:
| Etiqueta | Archivo                | Qué contiene                             |
| -------- | ---------------------- | ---------------------------------------- |
| `pos`    | `tweets_pos_clean.txt` | Tweets en español con polaridad positiva |
| `neg`    | `tweets_neg_clean.txt` | Tweets en español con polaridad negativa |
| `neu`    | `tweets_clean.txt`     | Tweets neutros o mezclados               |

2️⃣ Que el código no ha vuelto a descargar nada

El mensaje no muestra “Descargando …”, lo cual indica que:
- los archivos ya existían, o
- se han descargado correctamente en una ejecución anterior

y el cuaderno:
- ha comprobado que existen (os.path.exists)
- ha comprobado que no están vacíos (os.path.getsize)
- y ha decidido reutilizarlos

3️⃣ Qué representa el diccionario rutas

El bucle crea el diccionario `rutas` en memoria, y se usa en el siguiente apartado (11.2) para:
- leer los textos
- asignarles una etiqueta (pos, neg, neu)
- construir el DataFrame df_real

“Aquí no estamos analizando todavía nada. Solo estamos asegurándonos de que los datos reales existen localmente y sabemos dónde están.”

### 🧾 11.2 Cargar y preparar un subcorpus manejable (muestreo)

In [17]:
def leer_lineas(path, max_lines=None):
    lineas = []
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        for i, line in enumerate(f):
            line = line.strip()
            if line:
                lineas.append(line)
            if max_lines is not None and len(lineas) >= max_lines:
                break
    return lineas

# Para clase: limitamos el tamaño para que sea rápido y legible
N_POR_CLASE = 800  # ajusta si quieres más/menos

pos = leer_lineas(rutas["pos"], max_lines=N_POR_CLASE)
neg = leer_lineas(rutas["neg"], max_lines=N_POR_CLASE)
neu = leer_lineas(rutas["neu"], max_lines=N_POR_CLASE)

df_real = pd.DataFrame({
    "texto": pos + neg + neu,
    "etiqueta": (["pos"]*len(pos)) + (["neg"]*len(neg)) + (["neu"]*len(neu))
}).reset_index(drop=True)

print("Tamaño del subcorpus real:", len(df_real))
df_real.sample(8, random_state=7)



Tamaño del subcorpus real: 2400


,texto,etiqueta
1413,Perder una apuesta :(,neg
638,@Qrishnna jajaja eso no te lo creo :),pos
1183,#YoNoTengoPresidente ni amigos :(,neg
1665,FIEBRE DE NARCÓTICOS!! ARK: Survival Evolved #...,neu
2068,3 titulares de hoy que confirman la recuperaci...,neu
789,Kro kro currency devalue. Shabash :))),pos
1989,@Jhey_xx Hola mira mi canal en youtube espero ...,neu
1582,@simplemente_mv tu que me desperdicias :(,neg


**✅ BHP:**

📌 Qué ha hecho exactamente este código

1️⃣ Ha leído datos reales desde los tres ficheros > Las primeras líneas de pos / neg / neu significan:
- Lee hasta 800 tweets positivos
- Lee hasta 800 tweets negativos
- Lee hasta 800 tweets neutros/mixtos

👉 No se han creado textos artificiales.
👉 Son tweets reales, ya “limpios” en el repo original.

2️⃣ Ha construido un subcorpus equilibrado > `df_real` crea una tabla con:
- una columna texto → el contenido lingüístico
- una columna etiqueta → la clase de sentimiento original

Como hemos fijado N_POR_CLASE = 800: Tamaño del subcorpus real: 2400

3️⃣ El sample(8) muestra ejemplos reales y variados. Esto muestra claramente:
- Lenguaje informal
- Emojis y emoticonos
- Hashtags y menciones
- Ruido real (mayúsculas, inglés mezclado, spam)
- Polaridad no siempre obvia (muy buen punto para discusión)

“Hemos creado un subcorpus real, equilibrado y manejable, listo para análisis lingüístico.” Solo hemos hecho ingeniería de datos básica, que es parte esencial del PLN aplicado.
“Antes de analizar lenguaje, tenemos que decidir qué datos usamos, cuántos, y cómo los equilibramos. Aquí trabajamos con tweets reales, no con ejemplos inventados.”

### 🔍 11.3 Creación de un subcorpus temático (datos reales)

In [18]:
# Seleccionamos textos del corpus real que contienen referencias explícitas a género.
# Usamos (?: ...) para evitar el warning de "match groups".

import re

pat = re.compile(
    r"\b(?:mujer|mujeres|hombre|hombres|jefa|jefas|jefe|jefes|chica|chicas|chico|chicos)\b",
    re.IGNORECASE
)

df_sub = df_real[df_real["texto"].astype(str).str.contains(pat, regex=True)].copy()

# Comprobación rápida: tamaño y 3 ejemplos reales
print("Tamaño df_sub:", len(df_sub))
df_sub["texto"].head(3).tolist()


Tamaño df_sub: 35


['Se imaginan a los chicos agradeciendo por el premio con cara de orgullo?.Que bonito :).#MTVHottest One Direction',
 'Gracias @ElleAcIz , fue un concurso para realizar un videoclip sobre las mujeres en la ciencia :)',
 'La mujer se respeta anotala esa :)']

**✅ BHP**

📌 Qué ha hecho exactamente este código

1️⃣ Ha filtrado el subcorpus real por un criterio lingüístico > El patrón: `\b(?:mujer|mujeres|hombre|hombres|jefa|jefas|jefe|jefes|chica|chicas|chico|chicos)\b` significa:
- buscar menciones explícitas a términos de género
- en singular y plural
- respetando límites de palabra (\b)
- sin crear grupos de captura (?:...)

2️⃣ Qué significa Tamaño df_sub: 35 >> De los 2400 tweets reales del subcorpus general: solo 35 contienen menciones explícitas a esos términos

Esto no es un error. Es un resultado empírico importante.

Interpretación:
- las referencias explícitas a género son minoritarias en este dataset
- el fenómeno que quieres analizar no es ubicuo
- esto es exactamente lo que pasa con datos reales

3️⃣ Qué nos dicen los ejemplos > Los tres ejemplos:
- “Se imaginan a los chicos agradeciendo por el premio…” → referencia de género neutral/positiva, contexto mediático
- “Gracias … un concurso … sobre las mujeres en la ciencia” → discurso claramente valorativo y temático
- “La mujer se respeta anotala esa :)” → enunciado normativo / prescriptivo, muy interesante para análisis

👉 Estos ejemplos muestran:
- variedad pragmática
- carga valorativa implícita
- contextos distintos (entretenimiento, ciencia, moral)

“En corpus reales, los fenómenos de interés suelen ser raros. No inventamos datos: construimos subcorpus temáticos.”

“De 2400 tweets reales, solo 35 mencionan explícitamente estos términos de género. Esto no invalida el análisis; al contrario, muestra por qué necesitamos técnicas de PLN para localizar y estudiar fenómenos minoritarios.”

### 🧩 11.4 Aplicar el pipeline al subcorpus real

Reutilizamos las funciones y léxicos definidos antes:

- `normalizar_texto`
- `tokenizar_y_lematizar`
- `detectar_lexico_genero` (usa `lexico_genero`)
- `detectar_lexico_valorativo` (usa `dic_valorativo`)

Generamos una tabla con métricas por texto para poder comparar y discutir.


In [19]:
tokens_sub = [tokenizar_y_lematizar(t) for t in df_sub["texto"].tolist()]

resultados_sub = []
for i, texto_tokens in enumerate(tokens_sub):
    genero = detectar_lexico_genero(texto_tokens, lexico_genero)
    valorativo = detectar_lexico_valorativo(texto_tokens, dic_valorativo)

    # Score: suma de pesos del léxico valorativo encontrado
    score = sum(1 if dic_valorativo.get(t) == "positivo" else -1 for t in valorativo if t in dic_valorativo)

    resultados_sub.append({
        "texto_id": i,
        "etiqueta": df_sub.iloc[i]["etiqueta"],
        "texto": df_sub.iloc[i]["texto"],
        "tokens": texto_tokens,
        "genero_encontrado": genero,
        "valorativo_encontrado": valorativo,
        "n_genero": len(genero),
        "n_valorativo": len(valorativo),
        "score_valorativo": score
    })

df_resultados_sub = pd.DataFrame(resultados_sub)
df_resultados_sub.head()


,texto_id,etiqueta,texto,tokens,genero_encontrado,valorativo_encontrado,n_genero,n_valorativo,score_valorativo
0,0,pos,Se imaginan a los chicos agradeciendo por el p...,"[imaginar, chico, agradecer, premio, cara, bon...",[chico],[],1,0,0
1,1,pos,"Gracias @ElleAcIz , fue un concurso para reali...","[gracias, concurso, videoclip, mujer, ciencia]",[mujer],[],1,0,0
2,2,pos,La mujer se respeta anotala esa :),"[mujer, respetar, anotalir]",[mujer],[],1,0,0
3,3,pos,@jimenaduvois @pamelita888 Hola chicas!!!! :),"[hola, chica]",[chica],[],1,0,0
4,4,pos,¿Cuál es tu frase favorita? ¿Nos seguimos? Yo ...,"[frase, favorita, seguir, seguir, pasate, segu...",[hombre],[],1,0,0


**✅ BHP:**

📊 Qué representa esta tabla >> df_resultados_sub resume, para cada texto del subcorpus temático (35 tweets):
- qué tokens relevantes contiene,
- qué términos de género se han detectado,
- qué términos valorativos explícitos aparecen,
- y una métrica simple de carga valorativa.

Cada fila = un tweet real.

🧱 Interpretación columna por columna
- 🔹 texto_id = Un identificador interno: no tiene significado lingüístico, sirve para rastrear ejemplos concretos.
- 🔹 etiqueta = La etiqueta original del dataset: pos, neg, neu --> no es el resultado de tu análisis, sino un dato externo.
- 🔹 texto = El texto original del tweet, sin modificar: permite volver siempre al contexto real, imprescindible para interpretación cualitativa.
- 🔹 tokens = Los tokens (o lemas) tras normalización y stopwords: han desaparecido emojis, URLs, ruido; se conservan palabras con contenido semántico.
- 🔹 genero_encontrado = Lista de términos de género detectados: aquí suele aparecer un solo término porque el subcorpus se definió por menciones explícitas
- 🔹 valorativo_encontrado = Lista de términos valorativos explícitos detectados por el léxico: aquí aparece vacía ([]) en estos ejemplos. Esto es muy importante y no es un fallo.
- 🔹 n_genero = Número de términos de género detectados: en estos ejemplos: siempre 1 --> Es consistente con tweets cortos.
- 🔹 n_valorativo = Número de términos valorativos detectados: aquí: 0 --> Y esto nos lleva al punto clave.
- 🔹 score_valorativo = Definido como: score = len(valorativo) --> Por tanto: si no hay términos valorativos explícitos → 0. Es una medida muy conservadora

❗ Por qué valorativo_encontrado está vacío

Porque:
- el léxico valorativo es deliberadamente pequeño
- muchos tweets expresan valoración implícita, no léxica
- emoticonos (:), :() no cuentan como palabras
- frases normativas (“La mujer se respeta”) no usan adjetivos valorativos explícitos

👉 Este resultado enseña algo fundamental: El lenguaje evaluativo no siempre es léxico.

### 📊 11.5 Promedios por clase (pos/neg/neu)


Dos lecturas típicas:

- **Por etiqueta** (pos/neg/neu): ¿coincide el léxico valorativo con la etiqueta original?  
- **Por frecuencia**: ¿hay textos con muchas marcas de género o mucha carga valorativa?


In [20]:
resumen = df_resultados_sub.groupby("etiqueta")[["n_genero","n_valorativo","score_valorativo"]].mean().round(2)
resumen

,n_genero,n_valorativo,score_valorativo
etiqueta,,,
neg,0.89,0.00,0.00
neu,1.13,0.07,-0.07
pos,1.27,0.09,-0.09


**✅ BHP:**

📊 Qué mide exactamente esta tabla: La tabla resumen muestra, para cada clase original del dataset (pos, neg, neu), el promedio por texto de:
- n_genero: nº de términos de género detectados
- n_valorativo: nº de términos valorativos léxicos detectados
- score_valorativo: aquí es idéntico a n_valorativo

👉 Son medias sobre un subcorpus ya filtrado por género (35 textos).

- 🔹 n_genero = En tweets positivos, cuando aparece género, suele aparecer ligeramente más de un término por texto. En negativos, aparece algo menos. “Incluso con pocos datos, se pueden observar gradientes, no dicotomías.”
- 🔹 n_valorativo y score_valorativo = La gran mayoría de textos no contiene léxico valorativo explícito. Cuando aparece, es ligeramente más frecuente en positivos. En negativos: cero

❗ Por qué los negativos no tienen léxico valorativo > Porque:
- los tweets negativos suelen expresar emoción con emoticonos, ironía o contexto
- no necesariamente con adjetivos tipo “malo, horrible”
- el léxico usado es muy limitado
👉 Esto muestra el límite de los métodos basados en listas.

❗ Por qué los positivos sí aparecen un poco >> Porque:
- los positivos sí usan a veces adjetivos explícitos (bonito, genial)
- aunque sean pocos
- Pero el valor medio sigue siendo muy bajo.

“Un análisis léxico muy simple detecta muy poca valoración explícita, incluso en textos etiquetados como positivos o negativos.”

### 🔍 11.6 Ejemplos “interesantes” del subcorpus real

In [21]:
# A) Textos con más marcas de género
top_genero_sub = df_resultados_sub.sort_values(["n_genero","n_valorativo"], ascending=False).head(10)[
    ["etiqueta","texto","genero_encontrado","valorativo_encontrado","score_valorativo"]
]
top_genero_sub


,etiqueta,texto,genero_encontrado,valorativo_encontrado,score_valorativo
7,pos,Chicas si dan RT y ayudan consiguiendo mas a e...,"[chica, chica, chico]",[],0
6,pos,Chicas no se desanimen :) pronto las chicas la...,"[chica, chica]",[],0
21,neu,Científicos afirman que cuando las mujeres se ...,"[científico, mujer]",[],0
26,neu,Una #LeyDeMurphy dice: Un hombre con un reloj ...,"[hombre, hombre]",[],0
10,pos,Yo pienso que aunque uno este de novio puede h...,[chica],[malo],-1
31,neu,Hombres lindos *-* con cortes de cabello horri...,[hombre],[horrible],-1
0,pos,Se imaginan a los chicos agradeciendo por el p...,[chico],[],0
1,pos,"Gracias @ElleAcIz , fue un concurso para reali...",[mujer],[],0
2,pos,La mujer se respeta anotala esa :),[mujer],[],0
3,pos,@jimenaduvois @pamelita888 Hola chicas!!!! :),[chica],[],0


**✅ BHP:**

📌 Qué está mostrando exactamente esta tabla > `top_genero_sub` recoge los 10 textos del subcorpus temático que:
1. tienen más marcas de género (n_genero),
2. y, en caso de empate, más léxico valorativo (n_valorativo).

👉 No son “los más importantes”, sino los más densos en los rasgos que estamos analizando.

**🔍 Lectura fila a fila (patrones, no anécdotas)**

🟢 1. Repetición explícita de términos de género > Ejemplos claros:
- “Chicas si dan RT…” → [chica, chica, chico]
- “Chicas no se desanimen…” → [chica, chica]
- “Un hombre con un reloj…” → [hombre, hombre]

Interpretación: El género se usa como vocativo (“chicas…”) o como recurso retórico (repetición enfática), no necesariamente como tema de reflexión profunda
👉 Esto explica por qué n_genero puede ser alto sin que haya valoración explícita.

🟡 2. Presencia de valoración… muy puntual > Solo dos textos del top 10 contienen léxico valorativo explícito:
- “aunque uno esté de novio…” → malo
- “Hombres lindos… horribles” → horrible

Y aun así:
- el score es solo 1
- la valoración es adjetival y directa

Esto confirma lo visto antes: La valoración explícita es rara incluso en textos “ricos” en género.

🔵 3. Mezcla de etiquetas originales >> Observa las etiquetas:
- muchos pos
- varios neu
- casi ningún neg

Esto es importante: 👉 No hay una correspondencia directa entre:
- polaridad original del dataset
- densidad de género
- léxico valorativo explícito

### 🔎 11.7 Contexto (KWIC) en el subcorpus real

En redes sociales aparece un problema típico: si hacemos KWIC **sobre texto crudo** y buscamos una forma exacta (por ejemplo, `mujer`), podemos no recuperar casos en plural (`mujeres`) o variantes ortográficas.  
Para evitarlo sin complicar la vida, haremos KWIC **sobre los tokens/lemas** que ya hemos calculado en el pipeline (11.4). Así, las variantes suelen confluir en el mismo lema y el KWIC es más “robusto”.

> Nota: esto también ilustra un aprendizaje clave del PLN aplicado: el resultado depende mucho de *qué representación del texto* uses (crudo vs. normalizado vs. lematizado).


In [22]:
import pandas as pd

def kwic_tokens(tokens_corpus, term, window=5, max_lines=12):
    """KWIC simple sobre una lista de documentos tokenizados (cada doc = lista de tokens/lemas)."""
    lines = []
    for doc_id, toks in enumerate(tokens_corpus, start=1):
        for i, t in enumerate(toks):
            if t == term:
                left = " ".join(toks[max(0, i-window):i])
                right = " ".join(toks[i+1:i+1+window])
                lines.append({"texto_id": doc_id, "izq": left, "term": term, "dcha": right})
                if len(lines) >= max_lines:
                    return pd.DataFrame(lines)
    return pd.DataFrame(lines)

# Usamos los tokens/lemas del pipeline (df_resultados_sub)
tokens_corpus_sub = df_resultados_sub["tokens"].tolist()

terminos_prueba = ["mujer","hombre","jefa","jefe","chica","chico"]

for term in terminos_prueba:
    print("\n" + "="*70)
    print("KWIC (tokens/lemas) para:", term)
    kw = kwic_tokens(tokens_corpus_sub, term, window=6, max_lines=6)
    display(kw if len(kw) else "— sin ocurrencias (prueba con plural o revisa lematización) —")



KWIC (tokens/lemas) para: mujer


,texto_id,izq,term,dcha
0,2,gracias concurso videoclip,mujer,ciencia
1,3,,mujer,respetar anotalir
2,6,pendiente echeverrio,mujer,tunel casa sonar interesante
3,21,remilia,mujer,lcs abandonar competición acoso sufrido
4,22,científico afirmar,mujer,tomar foto grupo gravedad fuerte
5,23,china,mujer,soltero óvulo pertenecer



KWIC (tokens/lemas) para: hombre


,texto_id,izq,term,dcha
0,5,frase favorita seguir seguir pasate seguir,hombre,
1,13,cancion,hombre,llama cortar él vena
2,19,estúpido,hombre,
3,25,rato,hombre,milagro económico ver milagro español apretar él
4,27,leydemurphy,hombre,reloj hora hombre reloj seguro valgrind
5,27,leydemurphy hombre reloj hora,hombre,reloj seguro valgrind



KWIC (tokens/lemas) para: jefa


...

'— sin ocurrencias (prueba con plural o revisa lematización) —'


KWIC (tokens/lemas) para: chica


,texto_id,izq,term,dcha
0,4,hola,chica,
1,7,,chica,desanimar chica seguiran cuestión worthitvma
2,7,chica desanimar,chica,seguiran cuestión worthitvma
3,8,,chica,ayudar conseguir chica dar chico sorteo
4,8,chica ayudar conseguir,chica,dar chico sorteo
5,11,pensar novio hablar,chica,malo existir fidelidad



KWIC (tokens/lemas) para: chico


,texto_id,izq,term,dcha
0,1,imaginar,chico,agradecer premio cara bonito onir direction
1,8,chica ayudar conseguir chica dar,chico,sorteo
2,9,amar louis juro tranquilizarno estar,chico,onir direction
3,10,crew propuestaexa andreysucrewenlapropuestaexo,chico,talentoso
4,16,tanto gana abraza,chico,votar mtvhottest one direction
5,24,lleno risa aventura encantar viaje fugaz,chico,repetir tkmm


**✅ BHP:**

Qué significa este KWIC (antes de mirar los casos):
- Este KWIC ya no busca en el texto crudo, sino en los tokens/lemas que generó el pipeline (df_resultados_sub["tokens"]). Eso tiene dos efectos importantes:
- Normaliza (minúsculas, limpieza) y en parte lematiza (según spaCy esté o no).
- Evita el problema “mujer/mujeres” si el lematizador lo reduce a mujer, etc.
- Por eso ahora aparecen concordancias que antes te salían vacías.

**KWIC para mujer** > Aparece en varios documentos (texto_id 2, 3, 6, 21, 22, 23) y los contextos son variados:
- “gracias concurso videoclip mujer ciencia”: uso temático (mujeres en la ciencia), tono positivo/neutral.
- “mujer respetar …”: enunciado normativo (“la mujer se respeta”), valoración implícita (no necesariamente capturada por léxico valorativo).
- “científico afirmar mujer …”: enunciado tipo noticia/afirmación, registro más informativo.
- “... abandonar competición acoso sufrido”: contexto claramente social/problemático (acoso), muestra que mujer puede aparecer cerca de temas de violencia o discriminación.
- Otros casos (“mujer soltero óvulo…”): suenan a ruido/tema anecdótico o texto raro, típico de Twitter.

> Interpretación global: mujer aparece en el subcorpus tanto en usos neutros (informativos) como en usos normativos y en contextos de conflicto social. Esto es justo lo que un KWIC permite ver: no solo “aparece”, sino en qué tipo de enunciados.

**KWIC para hombre** > Aquí el patrón es distinto: Hay ocurrencias con contexto muy pobre por la derecha (vacío): eso suele pasar cuando hombre aparece al final del tweet o cerca del final del tramo tokenizado. Ejemplos con contenido llamativo:
- “estúpido hombre”: insulto directo (valoración negativa fuerte, aunque quizá el léxico valorativo no lo contenga).
- “cancion hombre llama cortar él vena”: contexto oscuro (posible referencia autolesiva o hiperbólica); sirve para explicar que Twitter mezcla registros y que el KWIC recupera contextos “crudos”.
- “leydemurphy … hombre … reloj …”: caso claro de repetición dentro del mismo tweet, por eso aparece dos veces con dos posiciones distintas; buen ejemplo para enseñar que el KWIC lista ocurrencias, no documentos.

> Interpretación global: hombre aparece con más frecuencia en registros coloquiales y evaluativos, con insulto o humor/aforismo (#LeyDeMurphy). Comparado con mujer, aquí se ve más discurso de “comentario” y menos “tema” (aunque con este tamaño pequeño no conviene generalizar).

**KWIC para jefa y jefe (sin ocurrencias)**  > Aquí el resultado es informativo: significa que en este df_sub (35 textos) no han aparecido esas formas exactas en los tokens/lemas. Las razones típicas (y didácticamente útiles) son:
- No están en los 35 textos filtrados (es lo más probable).
- O aparecen como plural (jefes/jefas) y aquí estás buscando singular.
- O el texto crudo tenía “@jefe…” o algo pegado que se perdió o cambió en el tokenizado.

> Interpretación global: no es un fallo del KWIC; es un recordatorio de que el subcorpus real, al ser pequeño, puede no contener todos los términos. Esto refuerza una idea de corpus: la ausencia también es un resultado.

** KWIC para chica** > Aquí aparecen patrones muy claros y repetitivos:
- “hola chica”: vocativo/llamada al interlocutor (uso conversacional).
- Repeticiones dentro del mismo tweet:
    - “chica desanimar chica seguiran …”
    - “chica ayudar conseguir chica dar chico sorteo …”
    - Esto muestra un fenómeno típico de redes: repetición apelativa (“chicas… las chicas…”).
- Un caso con valoración léxica explícita cerca: “pensar novio hablar chica malo …”: aquí aparece malo en el entorno, lo cual es un buen ejemplo para conectar KWIC con el léxico valorativo.

> Interpretación global: chica aparece como forma de apelación (“chicas…”) y en textos con dinámica de RT/sorteo, es decir, parte del “ecosistema” de Twitter (promos, llamadas a la acción). El KWIC te deja ver que no es solo “tema”, sino también función discursiva.

**KWIC para chico** > También es muy informativo:
- “imaginar chico agradecer premio …”: contexto positivo/celebratorio.
- “... dar chico sorteo”: de nuevo el patrón de promos/RT/sorteos.
- Varias menciones vinculadas a fandom y hashtags (One Direction, MTVHottest): el término chico aparece en contextos de cultura pop, no necesariamente “género” como tema político/social.

> Interpretación global: chico se usa mucho como sustantivo neutro para referirse a chicos en general (fandom, amigos, etc.), con fuerte ruido de cultura de Twitter.

**Lectura metodológica conjunta (lo más importante)**
- El KWIC ya funciona porque buscas en tokens/lemas: esto es una lección práctica clave sobre por qué preprocesar importa.
- Los términos no se comportan igual: mujer/hombre aparecen en contextos discursivos distintos; chica/chico aparecen muchísimo como vocativos/promos/fandom. Eso enseña que “término de género” no implica “discurso sobre género”.
- “Sin ocurrencias” para jefa/jefe es un resultado plausible por tamaño del subcorpus y por morfología (singular/plural). Buen momento para hablar de cobertura y diseño del subcorpus.

### 🤝 11.8 Co‑ocurrencias cerca de términos de género (subcorpus real)

Las co‑ocurrencias en Twitter suelen estar dominadas por “ruido” (URLs, abreviaturas, tokens muy cortos).  
Por eso aplicamos dos capas de filtrado:

1) **Stopwords estándar** (spaCy / NLTK) para español  
2) **Stopwords/ruido de Twitter** + filtro de longitud (para eliminar `rt`, `http`, `t`, etc.)

Esto no “arregla” el lenguaje real: solo ayuda a que la tabla sea más interpretable para discutir en clase.


In [24]:
import re
import pandas as pd
from collections import Counter

terminos_genero = ["mujer","mujeres","hombre","hombres","jefa","jefas","jefe","jefes","chica","chicas","chico","chicos"]

def tok_tweet(s):
    # Mantiene áéíóúüñ y descarta símbolos/emojis
    return re.findall(r"[a-záéíóúüñ]+", str(s).lower())

def coocurrencias_df(texts, target_terms, window_tokens=6, top_n=20, stopwords=None, extra_stop=None, min_len=3):
    targets = set(t.lower() for t in target_terms)
    stop = set(stopwords or [])
    stop |= set(extra_stop or [])
    counts = Counter()

    for text in texts:
        toks = tok_tweet(text)
        for i, w in enumerate(toks):
            if w in targets:
                start = max(0, i - window_tokens)
                end = min(len(toks), i + window_tokens + 1)
                for ctxt in toks[start:end]:
                    if ctxt in targets:
                        continue
                    if len(ctxt) < min_len:
                        continue
                    if ctxt in stop:
                        continue
                    counts[ctxt] += 1

    return pd.DataFrame(counts.most_common(top_n), columns=["coocurrencia", "frecuencia"])

# Stopwords estándar calculadas arriba (STOPWORDS_ES)
extra_stop_twitter = {"rt","dm","http","https","tco"}  # mínimo razonable

cooc_real_df = coocurrencias_df(
    df_sub["texto"].tolist(),
    terminos_genero,
    window_tokens=8,
    top_n=20,
    stopwords=STOPWORDS_ES,
    extra_stop=extra_stop_twitter,
    min_len=3
)

cooc_real_df if len(cooc_real_df) else "No se encontraron coocurrencias."


,coocurrencia,frecuencia
0,sigo,2
1,casa,2
2,desanimen,2
3,seguiran,2
4,ayudan,2
5,consiguiendo,2
6,dará,2
7,sorteo,2
8,mtvhottest,2
9,one,2


**✅ BHP:**

Qué está midiendo exactamente esta tabla > Esta tabla recoge palabras que aparecen cerca (±8 tokens) de términos de género (mujer, hombre, chica, chico…), después de:
- tokenizar tweets reales,
- eliminar stopwords estándar,
- eliminar stopwords típicas de Twitter,
- eliminar tokens muy cortos,
- y excluir los propios términos de género.

Por tanto, lo que ves son co-ocurrencias “limpias” y relativamente frecuentes en el subcorpus temático.

**Interpretación por grupos de co-ocurrencias**

1️⃣ Léxico de interacción y seguimiento: sigo, seguiran, ganas, ayudan, consiguiendo > Estas palabras son típicas de:
- llamadas a la acción (“síganme”, “ayuden”, “consigan…”),
- dinámicas de comunidad en Twitter.

> Interpretación: cuando aparecen términos como chica/chico, a menudo están integrados en discursos de interacción social (RT, follow, concursos), no en debates explícitos sobre género.

2️⃣ Léxico de sorteos y fandom: sorteo, mtvhottest, one, direction > Este grupo es muy claro:
- referencia a concursos, premios, fandoms (One Direction).

> Interpretación: gran parte de las menciones a chicos/chicas en este subcorpus no son ideológicas ni descriptivas, sino parte del lenguaje promocional y fan de Twitter.
Esto explica por qué el análisis de “género” en Twitter puede quedar dominado por ruido si no se afina el subcorpus.

3️⃣ Léxico narrativo/afectivo ligero: imaginan, agradeciendo, premio, cara, orgullo > Aquí aparecen verbos y sustantivos:
- asociados a emociones suaves y narración informal.

> Interpretación: algunos contextos donde aparece chico o chica son escenas imaginadas, celebratorias o emotivas, no evaluaciones explícitas (positivas/negativas) en términos del léxico valorativo.

4️⃣ Léxico cotidiano / anecdótico: casa, reloj, hora, simple > Este grupo muestra:
- temas cotidianos,
- frases hechas o comentarios sueltos.

> Interpretación: los términos de género se insertan en relatos muy cotidianos (“en casa”, “a tal hora”, “es simple”), lo que refuerza la idea de que la presencia del término no implica tematización del género.

**Lectura global (la clave para el aula)**

Las co-ocurrencias no apuntan mayoritariamente a discurso ideológico, sino a:
- interacción social,
- promociones,
- fandom,
- vida cotidiana.

Esto confirma algo importante que ya se intuía en 11.5:
- el n_genero mide presencia léxica, no “postura” ni “sesgo”.
- El hecho de que muchas co-ocurrencias tengan frecuencia 2 (y pocas tengan >2) refleja:
- el tamaño reducido del subcorpus (35 textos),
- y que estamos viendo patrones locales, no tendencias robustas.

## 💬 Preguntas de reflexión

Responde en **párrafos argumentados** (no listas telegráficas). Puedes incluir ejemplos breves si ayudan a justificar tu razonamiento.

### (A) Sobre 11.5 Promedios por clase (pos/neg/neu)
- ¿Qué tipos de valoración no estamos capturando con un enfoque basado en léxico?
- ¿Qué papel juegan los emojis en Twitter y por qué pueden “desaparecer” del análisis?
- ¿Qué problemas introduce la ironía para este tipo de método?
- ¿Qué cambiaría (y qué riesgos aparecerían) si usáramos un léxico mayor?

### (B) Sobre 11.6 Ejemplos “interesantes” del subcorpus
- ¿Qué estamos midiendo realmente con cada *etiqueta* (`pos/neg/neu`) y con nuestras métricas (`n_valorativo`, `score`, etc.)?
- ¿Por qué “chicas” aparece repetido? 
- ¿Es lo mismo mencionar género que hablar sobre género? 
- ¿Por qué un texto etiquetado como pos puede contener la palabra 'malo'? - ¿Qué tipos de valoración no estamos capturando?

### (C) Sobre 11.8 Co‑ocurrencias cerca de términos de género
- ¿Qué añadirías como stopwords específicas de Twitter? ¿Qué perderíamos si limpiamos demasiado?
- ¿Qué palabras del listado de co‑ocurrencias te parecen interpretables y por qué?
- ¿Dirías que este subcorpus habla sobre género?
- ¿Qué habría que cambiar para capturar discursos más políticos o problemáticos?

### Respuestas orientativas (para el profesorado)

#### 11.5 Promedios por clase (pos/neg/neu)

> ¿Qué tipos de valoración no estamos capturando con un enfoque basado en léxico?

El análisis que hacemos con un diccionario captura valoración **léxica explícita**: solo “ve” aquello que aparece como palabra dentro de una lista previa. Eso deja fuera mucha evaluación real: juicios expresados por el contexto, por presuposiciones, por construcciones sintácticas (comparaciones, concesiones) o por actos pragmáticos (reproches, insinuaciones). Por eso, que el promedio de `n_valorativo` sea bajo no significa que no haya valoración; significa que no coincide con el repertorio de palabras que hemos incluido en el diccionario.

> ¿Qué papel juegan los emojis en Twitter y por qué pueden “desaparecer” del análisis?

En Twitter, los **emojis** son un canal evaluativo clave. Pueden intensificar, suavizar o invertir la lectura de un enunciado, y a veces son el único marcador de actitud. Si el preprocesado los elimina (o los ignora el tokenizador), se pierde una parte importante de la señal y nuestras métricas tienden a infravalorar el componente afectivo del texto.

> ¿Qué problemas introduce la ironía para este tipo de método?

La **ironía** es un límite estructural para este enfoque: un texto puede contener palabras positivas y ser una crítica, o usar palabras negativas como broma afiliativa. Sin contexto conversacional, intención comunicativa o conocimiento compartido, un conteo léxico no puede resolver esto. Cuando la lectura humana y el score no coinciden, es una buena ocasión para discutir por qué “sentimiento” no se reduce a palabras sueltas.

> ¿Qué cambiaría (y qué riesgos aparecerían) si usáramos un léxico mayor?

Con un **léxico mayor** aumentaríamos cobertura (veríamos más coincidencias), pero también subiría el riesgo de falsos positivos: palabras evaluativas solo en determinados contextos, polisemia y ruido del registro. El trade‑off cobertura–precisión es parte del aprendizaje: ampliar el léxico obliga a justificar criterios y a validar con ejemplos.

#### 11.6 Ejemplos “interesantes” del subcorpus

> - ¿Qué estamos midiendo realmente con cada *etiqueta* (`pos/neg/neu`) y con nuestras métricas (`n_valorativo`, `score`, etc.)?

Las etiquetas `pos/neg/neu` vienen del dataset y reflejan una clasificación externa con criterios que no controlamos (y que puede incorporar emojis y lectura humana). Nuestras métricas, en cambio, miden presencia de términos detectados (léxico objetivo, léxico valorativo, co‑ocurrencias) y por tanto describen otra cosa. Si etiqueta y score no coinciden, eso no demuestra que una esté mal; muestra que estamos comparando dos operacionalizaciones distintas y que conviene ser explícitos sobre qué está midiendo cada variable.

> ¿Por qué “chicas” aparece repetido?

La repetición de chicas no indica necesariamente énfasis ideológico, sino que refleja usos discursivos propios de Twitter. En muchos casos, chicas funciona como vocativo colectivo (“chicas, no se desanimen…”, “chicas, ayuden…”), lo que lleva a que el término aparezca dos o más veces en el mismo mensaje. Además, en dinámicas de promoción o sorteos es habitual repetir el término para interpelar a la audiencia y maximizar la visibilidad. El KWIC permite observar este fenómeno claramente, algo que no se vería solo con conteos globales.

> ¿Es lo mismo mencionar género que hablar sobre género?

No. Los resultados muestran de forma clara que mencionar un término de género no equivale a tematizar el género. En el subcorpus real, chica o chico aparecen a menudo como formas de referencia neutra, apelativa o coloquial, sin que el texto trate cuestiones de desigualdad, identidad o conflicto de género. Hablar sobre género implica que el género sea el objeto del discurso, mientras que aquí, en muchos casos, es solo un recurso lingüístico dentro de interacciones sociales, fandoms o promociones.

> ¿Por qué un texto etiquetado como “pos” puede contener malo?

La etiqueta pos del dataset original se refiere al sentimiento global del tweet, no a cada palabra individual. Un texto puede contener una palabra negativa (malo) y, aun así, expresar una actitud general positiva (por ejemplo, ironía, contraste, negación o un balance global favorable). Esto ilustra una limitación fundamental del enfoque léxico: las palabras no tienen polaridad fija fuera de contexto, y la polaridad del texto no es simplemente la suma de sus términos.

> ¿Qué tipos de valoración no estamos capturando?

El análisis léxico no capta bien la ironía y el sarcasmo, muy frecuentes en Twitter. Tampoco las valoraciones implícitas, que se transmiten por contexto, orden de palabras o conocimiento compartido. Del mismo modo, no hemos capturado los emojis, los signos de puntuación enfática o las mayúsculas, que cargan mucha evaluación emocional. Finalmente, no se han capturado las valoraciones discursivas complejas, como estereotipos, presuposiciones o marcos narrativos. Esto explica por qué muchos textos parecen “neutros” según el léxico, aunque un lector humano perciba claramente una evaluación.

#### 11.8 Co‑ocurrencias

> ¿Qué añadirías como stopwords específicas de Twitter? ¿Qué perderíamos si limpiamos demasiado?

Las stopwords estándar (spaCy/NLTK) son generalistas; en Twitter suele ser útil añadir stopwords “de plataforma” (por ejemplo `rt`, `dm`, restos de URL como `http/https/tco` y tokens muy cortos) para que las co‑ocurrencias sean interpretables. Aun así, limpiar demasiado tiene costes: se pueden borrar marcas pragmáticas, estilos y hashtags que son parte del discurso. Limpiar es interpretar: hay que justificar qué se elimina y por qué.

> ¿Qué palabras del listado de co‑ocurrencias te parecen interpretables y por qué?

En co‑ocurrencias, lo relevante es argumentar qué parece “señal” y qué parece “ruido”. Una pauta didáctica útil es comparar la tabla antes y después de añadir stopwords específicas de Twitter y discutir qué se gana (menos artefactos) y qué se pierde (pistas de tono o de marco temático).

> ¿Dirías que este subcorpus habla sobre género?

En términos generales, no. El subcorpus contiene menciones a términos de género, pero las co-ocurrencias muestran que estos aparecen mayoritariamente en contextos de interacción social, sorteos, fandom y conversación cotidiana. El género no es el tema central del discurso, sino un elemento más del lenguaje informal de Twitter. Por tanto, el subcorpus habla con términos de género, pero no sobre género como problemática social o política.

> ¿Qué habría que cambiar para capturar discursos más políticos o problemáticos?

Habría que rediseñar el subcorpus. En concreto:
- Ampliar y afinar los términos de filtrado hacia conceptos como machismo, igualdad, acoso, feminismo, discriminación.
- Combinar términos de género con palabras clave de conflicto o política.
- Reducir el peso de discursos promocionales (por ejemplo, filtrando sorteos o fandoms).
- Aumentar el tamaño del subcorpus para capturar fenómenos menos frecuentes.
En otras palabras, no basta con cambiar el código: hay que cambiar la definición del fenómeno que queremos observar.


## 📋 Rúbrica de evaluación

⚖️ **Peso de esta actividad en la nota final:** trabajo práctico **60 %** · reflexión sobre 11.5, 11.6 y 11.8 **40 %** (13,3 % cada bloque)

🧭 **Cómo se evalúa:** se valora **el procedimiento, no solo el resultado**. Resuelve la actividad con el método trabajado en el cuaderno principal de la unidad. Organizar el código a tu manera, usar otros nombres de variables o resolverlo de forma más breve **no penaliza**, siempre que se reconozca ese método. Lo que sí penaliza es sustituirlo por librerías o herramientas que no hemos trabajado y que no explicas.

| **Criterio** | **Excelente (9-10)** | **Notable (7-8)** | **Aprobado (5-6)** | **Insuficiente (0-4)** |
|---|---|---|---|---|
| **Ejecución del código** | Todo el código se ejecuta correctamente sin errores; el flujo es coherente y reproducible. | El código se ejecuta completo y es reproducible, con algún ajuste menor que no afecta a los resultados. | El código funciona en general, pero presenta errores menores o ajustes manuales. | El código no se ejecuta o produce errores graves no resueltos. |
| **Claridad y organización del código** | Código bien estructurado, con nombres claros y comentarios que explican su función. | Código bien organizado y legible, con nombres claros, aunque los comentarios no cubran todos los bloques. | Código comprensible, aunque con organización mejorable o comentarios escasos. | Código desordenado, difícil de seguir o sin explicación. |
| **Construcción del subcorpus** | El subcorpus temático está bien definido, es suficiente en tamaño y se justifica su diseño. | El subcorpus está bien definido y es suficiente en tamaño, aunque su justificación podría desarrollarse más. | El subcorpus existe, pero es pequeño o poco afinado; justificación limitada. | El subcorpus es inapropiado, vacío o no se explica su construcción. |
| **Uso de tokenización / lematización** | Aplicación correcta y consciente del preprocesamiento; se entienden sus efectos. | Aplica correctamente el preprocesamiento y apunta algunos de sus efectos, sin llegar a valorarlos del todo. | El preprocesamiento funciona, pero sin reflexión clara sobre sus consecuencias. | Uso incorrecto o mecánico del preprocesamiento. |
| **Análisis léxico (conteos, métricas)** | Las métricas se calculan correctamente y se interpretan de forma precisa. | Las métricas se calculan correctamente y la interpretación es acertada, aunque no agota los resultados. | Las métricas se calculan, pero la interpretación es parcial o superficial. | Errores en el cálculo o interpretación incorrecta de las métricas. |
| **Interpretación del KWIC** | Se analizan los contextos con ejemplos claros y se extraen conclusiones lingüísticas. | Se analizan los contextos con ejemplos pertinentes y se esbozan conclusiones lingüísticas. | Se describen los ejemplos, pero con análisis limitado. | El KWIC se presenta sin interpretación o con conclusiones erróneas. |
| **Interpretación de co-ocurrencias** | Las co-ocurrencias se interpretan críticamente y se relacionan con el tipo de discurso. | Las co-ocurrencias se interpretan correctamente y se apunta su relación con el tipo de discurso. | Se comentan las co-ocurrencias, pero sin profundidad analítica. | No se interpretan o se confunden con ruido sin reflexión. |
| **Comprensión del sesgo y las limitaciones** | Análisis crítico profundo de lo que el método mide y de lo que deja fuera. | Identifica con claridad qué mide el método y qué deja fuera, con un desarrollo aún breve. | Comprensión básica de las limitaciones, sin desarrollo. | Confusión conceptual sobre qué mide el análisis. |
| **Reflexión metodológica** | Se discuten decisiones analíticas (términos, limpieza, tamaño) y sus efectos. | Se discuten las decisiones analíticas principales y se apunta su efecto, sin desarrollarlo del todo. | Se mencionan decisiones, pero sin evaluar su impacto. | No hay reflexión metodológica. |
| **Reflexión final** | Argumentada, matizada y conectada con los resultados obtenidos. | Argumentada y conectada con los resultados obtenidos, aunque con menos matices. | Presente, pero superficial o poco conectada con el análisis. | Ausente o meramente descriptiva. |


## 🧾 RÚBRICA DE EVALUACIÓN (PROFESORADO)

Preguntas de reflexión – Apartados 11.5, 11.6 y 11.8

Objeto de evaluación:
Capacidad del alumnado para interpretar críticamente los resultados del análisis léxico y comprender las limitaciones metodológicas del enfoque aplicado a datos reales de Twitter.

🅐 Reflexión sobre 11.5 – Promedios por clase (pos/neg/neu)

| **Criterio** | **Excelente (9-10)** | **Notable (7-8)** | **Aprobado (5-6)** | **Insuficiente (0-4)** |
|---|---|---|---|---|
| **Comprensión de lo que NO captura el léxico** | Identifica con claridad valoraciones implícitas, discursivas, pragmáticas o contextuales, explicando por qué quedan fuera del método. | Identifica y tipifica algunas valoraciones no capturadas, aunque no explique del todo por qué quedan fuera del método. | Reconoce que hay valoraciones no capturadas, pero sin tipificarlas claramente. | Confunde el léxico con “todo el significado” o no identifica límites claros. |
| **Análisis del papel de los emojis** | Explica el rol comunicativo de los emojis en Twitter y justifica por qué desaparecen del análisis automático. | Explica el papel comunicativo de los emojis y apunta por qué se pierden, sin detallar el proceso. | Menciona los emojis como relevantes, pero sin explicar bien su exclusión o función. | Ignora el papel de los emojis o los trata como irrelevantes. |
| **Comprensión del problema de la ironía** | Describe la ironía como un problema estructural del enfoque léxico y da ejemplos plausibles. | Describe la ironía como un problema del enfoque léxico, aunque sin ejemplos o sin presentarla como límite estructural. | Reconoce que la ironía “da problemas”, pero sin profundizar. | No identifica la ironía como un problema o la confunde con polaridad negativa. |
| **Reflexión sobre ampliar el léxico** | Analiza ventajas y riesgos (ruido, polisemia, falsos positivos) de usar un léxico mayor. | Analiza las ventajas de un léxico mayor y menciona algún riesgo, sin desarrollarlos. | Señala que un léxico mayor cambiaría resultados, pero sin discutir riesgos. | Asume que “más léxico = mejor” sin reflexión crítica. |


🅑 Reflexión sobre 11.6 – Ejemplos “interesantes” del subcorpus

| **Criterio** | **Excelente (9-10)** | **Notable (7-8)** | **Aprobado (5-6)** | **Insuficiente (0-4)** |
|---|---|---|---|---|
| **Qué miden las etiquetas y las métricas** | Distingue las etiquetas externas `pos/neg/neu` del dataset de las métricas calculadas (`n_valorativo`, `score_valorativo`, presencia de términos) y explica que operacionalizan cosas distintas. | Distingue etiquetas y métricas y señala que miden cosas distintas, sin precisar del todo cuáles. | Distingue parcialmente etiquetas y métricas, con poca precisión. | Confunde la etiqueta del dataset con el resultado del análisis léxico. |
| **Repetición de «chicas»** | Explica la repetición como vocativo colectivo, llamada a la acción, dinámica de Twitter, sorteos, fandom o estrategia de visibilidad, apoyándose en el KWIC. | Explica la repetición como uso discursivo propio de Twitter, con algún ejemplo, sin cubrir varias causas. | Da una explicación plausible, pero poco conectada con el corpus. | Interpreta la repetición como sesgo o tema de género sin matiz. |
| **Mencionar género frente a hablar sobre género** | Diferencia con claridad la presencia léxica de términos de género y la tematización del género como problema social o discursivo, con ejemplos del subcorpus. | Diferencia mención y tematización, y lo justifica con algún ejemplo del subcorpus. | Reconoce la diferencia, pero con justificación limitada. | Afirma que toda mención de género equivale a discurso sobre género. |
| **Texto «pos» que contiene «malo»** | Explica que la etiqueta `pos` es global y contextual, que una palabra negativa aislada no determina el sentido, y que quedan fuera ironía, contexto, emojis o valoración implícita. | Explica que la etiqueta es global y que una palabra aislada no determina el sentido, sin desarrollar qué queda fuera. | Explica que puede haber palabras negativas en textos positivos, pero con pocos matices. | Asume contradicción automática o error del dataset sin argumentar. |

🅒 Reflexión sobre 11.8 – Co-ocurrencias cerca de términos de género

| **Criterio** | **Excelente (9-10)** | **Notable (7-8)** | **Aprobado (5-6)** | **Insuficiente (0-4)** |
|---|---|---|---|---|
| **Reflexión sobre stopwords de Twitter** | Propone stopwords específicas justificadas y explica el riesgo de sobrelimpieza. | Propone stopwords específicas justificadas y menciona el riesgo de sobrelimpieza, sin ejemplificar qué se perdería. | Propone algunas stopwords, pero sin discutir pérdidas. | Elimina términos sin criterio o no reflexiona sobre limpieza. |
| **Interpretación de co-ocurrencias** | Identifica palabras interpretables y las relaciona con tipos de discurso concretos. | Identifica palabras interpretables y las relaciona con algún tipo de discurso, sin concretarlo del todo. | Comenta co-ocurrencias de forma descriptiva. | Enumera palabras sin interpretación. |
| **Evaluación del tema del subcorpus** | Concluye razonadamente si el subcorpus habla *sobre* género o solo lo menciona. | Concluye correctamente si el subcorpus habla sobre género, con una justificación breve. | Da una respuesta general sin justificar con resultados. | Respuesta afirmativa o negativa sin análisis. |
| **Propuestas de mejora metodológica** | Propone cambios coherentes (términos, filtros, rediseño del subcorpus) para captar discursos políticos o problemáticos. | Propone cambios coherentes y concretos, aunque sin cubrir todos los frentes. | Sugiere cambios, pero sin concreción metodológica. | No propone mejoras o las plantea de forma ingenua. |

📌 Orientación global de corrección

- Excelente: reflexión crítica, consciente de los límites del método y basada en los resultados.
- Notable: reflexión correcta y conectada con los resultados, con algún límite del método sin desarrollar.

- Aprobado: comprensión general correcta, pero con análisis superficial o incompleto.

- Insuficiente: confusión conceptual, respuestas genéricas o ausencia de reflexión.

## 📚 4. Recursos adicionales

Estos recursos sirven para profundizar en los métodos utilizados en la práctica. No es necesario dominarlos todos para aprobar.

🔤 Léxico, polaridad y análisis basado en diccionarios

- Bing Liu (2012) – Sentiment Analysis and Opinion Mining (introducción clara al análisis de sentimiento basado en léxicos).
- Hu & Liu (2004) – Mining and Summarizing Customer Reviews (texto clásico sobre polaridad léxica).

🧠 Corpus, contexto y discurso

- Baker, P. (2006) – Using Corpora in Discourse Analysis (excelente para entender por qué los conteos no “hablan solos”).
- McEnery & Hardie (2012) – Corpus Linguistics (capítulos sobre KWIC y concordancias).

🧰 Herramientas y documentación

- spaCy (español): https://spacy.io/models/es (para entender lematización y stopwords).
- NLTK Book (capítulos 1–3): https://www.nltk.org/book/ (introducción muy accesible a tokenización y co-ocurrencias).

🐦 Lenguaje y redes sociales

- Eisenstein (2013) – What to do about bad language on the internet (texto clave sobre ruido, variación y lenguaje informal).
- Zappavigna (2012) – Discourse of Twitter and Social Media (para entender por qué Twitter no se comporta como la prensa).